# Stichprobe ohne die großen Datenbereitsteller

Zieht eine zweite Stichprobe ($n = 50$), die **ausschließlich** Datensätze der kleineren
Portale enthält. Zweck ist eine Robustheitsprüfung: Halten die Ergebnisse der
Kernstichprobe auch dort, wo nicht die aufkommensstärksten Portale dominieren?

**Abgrenzung „groß".** Als groß gelten genau die Bereitsteller, die in der Kernstichprobe
(Notebook 12) als eigene Sekundärschicht geführt werden — die je vier aufkommensstärksten
Bereitsteller innerhalb der Geo- und der Nicht-Geo-Gruppe. Ihre Vereinigung wird hier
**global** ausgeschlossen, also aus beiden Gruppen. Das ist bewusst strenger als das
`rest`-Stratum der Kernstichprobe: Dort kann ein Bereitsteller, der nur in einer der
beiden Gruppen zu den Top~4 zählt, über die andere Gruppe weiterhin einfließen.

**Schichtung.** Innerhalb des verbleibenden Pools wird dieselbe zweistufige Logik
angewendet wie in der Kernstichprobe: primär Geo/Nicht-Geo, sekundär die vier
aufkommensstärksten der verbliebenen Bereitsteller je Gruppe zuzüglich einer
Restkategorie, fünf Datensätze je Stratum. Ohne diese zweite Stufe würde ein einzelner
mittelgroßer Bereitsteller einen erheblichen Teil der Stichprobe stellen.

Die Zellen zum Laden des Snapshots und zur Geo-Klassifikation sind unverändert aus
Notebook 12 übernommen, damit beide Stichproben auf identischer Klassifikation beruhen.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
SNAPSHOT_FILE         = "data/raw/govdata_ckan_snapshot2026-05-15_09-57-31.jsonl"
RANDOM_SEED           = 67
SAMPLE_SIZE           = 50
SAMPLES_PER_SECONDARY = 5     # je Sekundaerschicht

# Anzahl der je Primaerschicht als "gross" eingestuften Bereitsteller. Ihre
# Vereinigung wird global ausgeschlossen (siehe Einleitung).
TOP_N_GROSS = 4

# Bereits in der Kernstichprobe gezogene Datensaetze ausschliessen, damit beide
# Stichproben disjunkt sind. None = keine Ausschlussliste.
KERNSTICHPROBE_MANIFEST = "../../data/sample_2026-06-25_09-57/sample_manifest.csv"

# Zielordner-Praefix (Zeitstempel wird angehaengt).
OUT_PREFIX = "sample_small_portals"
# ============================================================================

In [ ]:
import json
from pathlib import Path

import pandas as pd

snapshot_path = Path(SNAPSHOT_FILE)
if not snapshot_path.exists():
    raise FileNotFoundError(f"{snapshot_path} not found.")

records = [
    json.loads(line)
    for line in snapshot_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
df = pd.json_normalize(records)
print(f"Loaded:              {len(df):,} datasets")


def _has_netcdf(row) -> bool:
    for res in (row.get("resources") or []):
        if not isinstance(res, dict):
            continue
        fmt  = (res.get("format")   or "").lower()
        url  = (res.get("url") or res.get("download_url") or res.get("access_url") or "").lower()
        mime = (res.get("mimetype") or "").lower()
        if "netcdf" in fmt or ".nc" in fmt:
            return True
        if url.endswith(".nc"):
            return True
        if "netcdf" in mime:
            return True
    return False


df = df[~df.apply(_has_netcdf, axis=1)].reset_index(drop=True)
print(f"After NetCDF filter: {len(df):,} datasets")

In [ ]:
import re

GEO_FORMAT_PATTERNS = {
    "WMS":        [r"\bWMS\b", r"WMS_SRVC", r"OGC/WMS", r"application/vnd\.ogc\.wms_xml", r"SERVICE=WMS"],
    "WFS":        [r"\bWFS\b", r"WFS_SRVC", r"OGC/WFS", r"application/vnd\.ogc\.wfs_xml", r"SERVICE=WFS"],
    "WCS":        [r"\bWCS\b", r"WCS_SRVC", r"SERVICE=WCS"],
    "WMTS":       [r"\bWMTS\b", r"WMTS_SRVC", r"application/vnd\.ogc\.wmts_xml", r"SERVICE=WMTS"],
    "GML":        [r"\bGML\b", r"application/gml\+xml", r"application/vnd\.ogc\.gml", r"GML Application Schema", r"INSPIRE-GML", r"XML/GML", r"gml\+xml"],
    "GeoJSON":    [r"\bGEOJSON\b", r"\bGeoJSON\b", r"application/vnd\.geo\+json", r"application/geo\+json", r"geo\+json"],
    "Shapefile":  [r"\bSHP\b", r"\bSHAPE\b", r"\bShape\b", r"Shapefile", r"Shape File", r"ESRI[- ]?Shape", r"gezippte shape", r"gezippte shapedatei", r"\bshp\b"],
    "GeoPackage": [r"\bGPKG\b", r"GeoPackage", r"Geopackage", r"geopackage", r"application/geopackage\+sqlite3", r"application/gpkg"],
    "KML/KMZ":    [r"\bKML\b", r"\bKMZ\b"],
    "GeoTIFF":    [r"\bGEOTIFF\b", r"GeoTIFF", r"GeoTIF", r"Geo-TIFF", r"Cloud Optimized GeoTIFF", r"\bCOG\b", r"georef\. TIFF"],
    "DXF/DWG":    [r"\bDXF\b", r"\bDWG\b", r"dwg/dxf"],
    "CityGML":    [r"CityGML", r"CITYGML", r"citygml"],
    "NAS":        [r"\bNAS\b", r"\bnas\b"],
    "GPX":        [r"\bGPX\b", r"\bgpx\b"],
    "WKT":        [r"\bWKT\b"],
    "LAS/LAZ":    [r"\bLAS\b", r"\bLAZ\b", r"LAS/LAZ"],
    "MapInfo":    [r"MapInfo", r"\bMIF\b", r"\bTAB\b"],
    "ArcGIS/ESRI":[r"ESRI Hub", r"esri hub", r"ArcGIS", r"arcgis", r"File ?Geodatabase", r"Geodatabase", r"GDB", r"MDB", r"SDE-Feature-Class"],
    "GeoRaster":  [r"GRID_ASCII", r"ASCII-Grid", r"XYZ-ASCII", r"\bGRIB\b", r"\bBUFR\b"],
}


def _classify_geo(fmt: str):
    fmt = (fmt or "").strip()
    if not fmt:
        return None
    for group, patterns in GEO_FORMAT_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, fmt, re.IGNORECASE):
                return group
    return None


geo_rows = []
for idx, row in df.iterrows():
    for res in (row.get("resources") or []):
        if not isinstance(res, dict):
            continue
        geo_group = _classify_geo(str(res.get("format") or ""))
        if geo_group:
            geo_rows.append({"dataset_index": idx, "geo_format_group": geo_group})
            break  # one match per dataset is enough to mark it as geo

geo_distribution_rows = pd.DataFrame(geo_rows)
geo_dataset_indices = (
    set(geo_distribution_rows["dataset_index"].unique())
    if not geo_distribution_rows.empty
    else set()
)
print(f"Geo datasets:     {len(geo_dataset_indices):,}")
print(f"Non-geo datasets: {len(df) - len(geo_dataset_indices):,}")

In [ ]:
# --- Sampling-Frame aufbauen (identisch zu Notebook 12) ---------------------
import hashlib
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests


def _publisher(row: pd.Series) -> str:
    for src in [
        row.get("organization.name"),
        (row.get("organization") or {}).get("name") if isinstance(row.get("organization"), dict) else None,
        (row.get("organization") or {}).get("title") if isinstance(row.get("organization"), dict) else None,
        row.get("maintainer"),
        row.get("author"),
    ]:
        if isinstance(src, str) and src.strip():
            return src.strip()
    return ""


geo_indices = set(geo_dataset_indices)
frame_rows = []
for idx, row in df.iterrows():
    name = row.get("name")
    if not isinstance(name, str) or not name.strip():
        continue
    pub = _publisher(row)
    if not pub:
        continue
    frame_rows.append({
        "dataset_index": idx,
        "dataset_id": row.get("id"),
        "dataset_name": name.strip(),
        "publisher": pub,
        "primary_stratum": "geo" if idx in geo_indices else "non_geo",
    })

frame = pd.DataFrame(frame_rows)
PRIMARY_STRATA = ["geo", "non_geo"]
print(f"Frame: {len(frame):,} Datensaetze, {frame.publisher.nunique()} Bereitsteller")

# --- Grosse Bereitsteller bestimmen und global ausschliessen ----------------
top_map = {
    pr: frame[frame.primary_stratum == pr]["publisher"].value_counts().head(TOP_N_GROSS).index.tolist()
    for pr in PRIMARY_STRATA
}
for pr, v in top_map.items():
    print(f"  Top-{TOP_N_GROSS} {pr:8s}: {v}")

gross = sorted(set().union(*top_map.values()))
print(f"\nGlobal ausgeschlossen ({len(gross)}): {gross}")

pool = frame[~frame["publisher"].isin(gross)].copy()
print(f"Rest-Pool: {len(pool):,} Datensaetze, {pool.publisher.nunique()} Bereitsteller")

# --- Ueberschneidung mit der Kernstichprobe entfernen -----------------------
if KERNSTICHPROBE_MANIFEST:
    mp = Path(KERNSTICHPROBE_MANIFEST)
    if mp.exists():
        kern = pd.read_csv(mp)
        vorher = len(pool)
        pool = pool[~pool["dataset_id"].isin(set(kern["dataset_id"]))]
        print(f"Nach Abzug der Kernstichprobe: {len(pool):,} ({vorher - len(pool)} entfernt)")
    else:
        print(f"HINWEIS: Manifest nicht gefunden ({mp}) - keine Ueberschneidung entfernt.")

print()
print(pool.groupby("primary_stratum").size().to_string())

In [ ]:
# --- Sekundaerschichtung innerhalb des Rest-Pools und Ziehung ---------------
top_rest = {
    pr: pool[pool.primary_stratum == pr]["publisher"].value_counts().head(4).index.tolist()
    for pr in PRIMARY_STRATA
}
pool["secondary_stratum"] = pool.apply(
    lambda r: r["publisher"] if r["publisher"] in top_rest[r["primary_stratum"]] else "rest",
    axis=1,
)

selected_parts = []
for primary in PRIMARY_STRATA:
    sub = pool[pool.primary_stratum == primary]
    print(f"Ziehung {primary.upper()}:")
    for sec in top_rest[primary] + ["rest"]:
        stratum = sub[sub.secondary_stratum == sec]
        n = min(SAMPLES_PER_SECONDARY, len(stratum))
        if n == 0:
            continue
        # Stratum-spezifischer Seed: reproduzierbar und unabhaengig je Schicht.
        key = f"{RANDOM_SEED}-small-{primary}-{sec}"
        seed = int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**31 - 1)
        selected_parts.append(stratum.sample(n=n, random_state=seed))
        print(f"  {sec:34s} -> {n:2d}  (Pool {len(stratum):,})")

selected = pd.concat(selected_parts, ignore_index=True)

# Auffuellen bzw. kuerzen, falls einzelne Straten zu klein waren.
if len(selected) < SAMPLE_SIZE:
    uebrig = pool[~pool.dataset_index.isin(set(selected.dataset_index))]
    if not uebrig.empty:
        fill = uebrig.sample(n=min(SAMPLE_SIZE - len(selected), len(uebrig)), random_state=RANDOM_SEED).copy()
        fill["secondary_stratum"] = fill["secondary_stratum"].astype(str) + "_fill"
        selected = pd.concat([selected, fill], ignore_index=True)
elif len(selected) > SAMPLE_SIZE:
    selected = selected.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\nStichprobe: {len(selected)} Datensaetze (Seed {RANDOM_SEED})")
print(selected.groupby(["primary_stratum", "secondary_stratum"]).size().to_string())

In [ ]:
# --- RDF/XML abrufen und speichern (Fetch-Logik aus Notebook 12) ------------
def _create_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": "govdata-stratified-sampler/0.1",
                      "Accept": "application/rdf+xml, application/xml;q=0.9, */*;q=0.1"})
    return s


def _sanitize(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "-", (value or "").strip())
    return value.strip("-") or "dataset"


def _fetch_rdf_xml(session: requests.Session, name: str, timeout: int = 60) -> bytes:
    for pattern in [
        "https://www.govdata.de/dataset/{n}.rdf",
        "https://www.govdata.de/ckan/dataset/{n}.rdf",
        "https://govdata.de/dataset/{n}.rdf",
        "https://govdata.de/ckan/dataset/{n}.rdf",
        "https://www.govdata.de/dataset/{n}.xml",
        "https://www.govdata.de/ckan/dataset/{n}.xml",
        "https://govdata.de/dataset/{n}.xml",
        "https://govdata.de/ckan/dataset/{n}.xml",
    ]:
        try:
            r = session.get(pattern.format(n=name), timeout=timeout, allow_redirects=True)
        except requests.RequestException:
            continue
        if r.status_code != 200:
            continue
        ct = (r.headers.get("Content-Type") or "").lower()
        if b"<rdf:rdf" in r.content[:1000].lower() or "application/rdf+xml" in ct or "xml" in ct:
            return r.content
    raise RuntimeError(f"Kein RDF/XML-Endpunkt erreichbar fuer '{name}'.")


timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
out_dir = Path("../../data") / f"{OUT_PREFIX}_{timestamp}"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Zielordner: {out_dir}\n")

session = _create_session()
saved, failed, counter = [], [], {}

for i, (_, row) in enumerate(selected.iterrows(), 1):
    primary = row["primary_stratum"]
    sec = _sanitize(str(row["secondary_stratum"]).lower())
    name = row["dataset_name"]
    key = (primary, sec)
    counter[key] = counter.get(key, 0) + 1
    seq = counter[key]
    fname = f"{primary}_{sec}.rdf" if seq == 1 else f"{primary}_{sec}_{seq:02d}.rdf"
    try:
        (out_dir / fname).write_bytes(_fetch_rdf_xml(session, name))
        saved.append({**row.to_dict(), "file_path": str(out_dir / fname)})
        print(f"[{i:2d}/{len(selected)}] OK   {fname}")
    except Exception as exc:
        failed.append({"dataset_name": name, "error": str(exc)})
        print(f"[{i:2d}/{len(selected)}] FAIL {name} ({exc})")

pd.DataFrame(saved).to_csv(out_dir / "sample_manifest.csv", index=False)
print(f"\nGespeichert: {len(saved)} | Fehlgeschlagen: {len(failed)}")
print(f"Manifest: {out_dir / 'sample_manifest.csv'}")

In [ ]:
# --- Ausfaelle nachziehen ---------------------------------------------------
# Nicht jeder Datensatz des CKAN-Snapshots ist ueber govdata.de auch als RDF
# abrufbar; bei einzelnen Bereitstellern (z. B. statistisches-bundesamt) scheitert
# der Abruf sogar ueberwiegend. Ohne diesen Schritt bliebe die Stichprobe kleiner
# als SAMPLE_SIZE und waere zwischen den Straten unausgewogen.
#
# Vorgehen: je unterbesetztem Stratum werden Ersatzkandidaten aus demselben
# Stratum gezogen (mit Reserve fuer weitere Ausfaelle). Laesst sich ein Stratum
# so nicht fuellen, wird aus der Restkategorie derselben Primaerschicht ersetzt
# und die Ersetzung im Manifest als "<stratum>_ersatz_<quelle>" gekennzeichnet.

RESERVE_FAKTOR = 6   # so viele Kandidaten je fehlendem Datensatz vorhalten

versucht = set(selected["dataset_id"]) | {s["dataset_id"] for s in saved}
bestand = pd.DataFrame(saved).groupby(
    ["primary_stratum", "secondary_stratum"]).size().to_dict() if saved else {}
nachgezogen = []


def _freier_name(primary, sec_label):
    """Naechster freier Dateiname im Stratum.

    Wichtig: Die Nummerierung wird aus den tatsaechlich vorhandenen Dateien
    abgeleitet, nicht aus einem Zaehler. Der erste Abruf oben zaehlt Versuche
    (inklusive Fehlschlaege), ein Zaehler ueber Erfolge wuerde hier deshalb
    bereits vergebene Namen erneut vergeben und Dateien ueberschreiben.
    """
    stem = f"{primary}_{_sanitize(sec_label.lower())}"
    if not (out_dir / f"{stem}.rdf").exists():
        return f"{stem}.rdf"
    k = 2
    while (out_dir / f"{stem}_{k:02d}.rdf").exists():
        k += 1
    return f"{stem}_{k:02d}.rdf"


def _hole(row, primary, sec_label):
    fname = _freier_name(primary, sec_label)
    (out_dir / fname).write_bytes(_fetch_rdf_xml(session, row["dataset_name"]))
    return fname


for primary in PRIMARY_STRATA:
    for sec in top_rest[primary] + ["rest"]:
        fehlt = SAMPLES_PER_SECONDARY - bestand.get((primary, sec), 0)
        if fehlt <= 0:
            continue
        kand = pool[(pool.primary_stratum == primary) & (pool.secondary_stratum == sec)
                    & (~pool.dataset_id.isin(versucht))]
        seed = int(hashlib.md5(f"topup-{primary}-{sec}".encode()).hexdigest(), 16) % (2**31 - 1)
        kand = kand.sample(n=min(len(kand), fehlt * RESERVE_FAKTOR), random_state=seed)
        print(f"{primary}/{sec}: {fehlt} fehlen, {len(kand)} Kandidaten")
        for _, row in kand.iterrows():
            if fehlt <= 0:
                break
            versucht.add(row["dataset_id"])
            try:
                fname = _hole(row, primary, sec)
            except Exception:
                continue
            fehlt -= 1
            nachgezogen.append({**row.to_dict(), "file_path": str(out_dir / fname)})
            print(f"   OK   {fname}")
        if fehlt > 0:
            # Fallback: Restkategorie derselben Primaerschicht.
            print(f"   {fehlt} nicht ersetzbar -> Ersatz aus der Restkategorie")
            ersatz = pool[(pool.primary_stratum == primary) & (pool.secondary_stratum == "rest")
                          & (~pool.dataset_id.isin(versucht))]
            ersatz = ersatz.sample(n=min(len(ersatz), fehlt * RESERVE_FAKTOR),
                                   random_state=seed + 1)
            for _, row in ersatz.iterrows():
                if fehlt <= 0:
                    break
                versucht.add(row["dataset_id"])
                try:
                    fname = _hole(row, primary, "rest")
                except Exception:
                    continue
                fehlt -= 1
                eintrag = row.to_dict()
                eintrag["secondary_stratum"] = f"rest_ersatz_{sec}"
                nachgezogen.append({**eintrag, "file_path": str(out_dir / fname)})
                print(f"   OK   {fname}  (Ersatz fuer {sec}, {row['publisher']})")

if nachgezogen:
    alle = pd.concat([pd.DataFrame(saved), pd.DataFrame(nachgezogen)], ignore_index=True)
    alle.to_csv(out_dir / "sample_manifest.csv", index=False)
else:
    alle = pd.DataFrame(saved)

print(f"\nNachgezogen: {len(nachgezogen)} | Stichprobe gesamt: {len(alle)}")
print(alle.groupby(["primary_stratum", "secondary_stratum"]).size().to_string())